<a href="https://colab.research.google.com/github/kirti73/AI_lab/blob/main/WEEK8/1BF24CS140_WEEK8_FORWARDCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementation of First Order Logic (FOL) Forward Chaining

Below, we define a standard Python representation for terms, facts (atomic sentences), and rules (implications). We then implement a unification and standardization process followed by the forward chaining algorithm to prove `Criminal(Robert)` based on the provided facts.

In [ ]:
import copy

class Term:
    def __init__(self, name, is_variable=False):
        self.name = name
        self.is_variable = is_variable

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Term) and self.name == other.name and self.is_variable == other.is_variable

    def __hash__(self):
        return hash((self.name, self.is_variable))

class Predicate:
    def __init__(self, name, terms):
        self.name = name
        self.terms = terms # List of Term objects

    def __repr__(self):
        return f"{self.name}({', '.join(map(str, self.terms))})"

    def __eq__(self, other):
        return isinstance(other, Predicate) and self.name == other.name and self.terms == other.terms

    def __hash__(self):
        return hash((self.name, tuple(self.terms)))

class Rule:
    def __init__(self, premises, conclusion):
        self.premises = premises # List of Predicate objects
        self.conclusion = conclusion # Single Predicate object

    def __repr__(self):
        premises_str = " ^ ".join(map(str, self.premises))
        return f"{premises_str} => {self.conclusion}"

# Helper to standardize variables in a rule
var_counter = 0
def standardize_variables(rule):
    global var_counter
    var_counter += 1
    subst = {}

    def get_new_term(term):
        if not term.is_variable:
            return term
        if term.name not in subst:
            subst[term.name] = Term(f"{term.name}_{var_counter}", is_variable=True)
        return subst[term.name]

    new_premises = []
    for p in rule.premises:
        new_premises.append(Predicate(p.name, [get_new_term(t) for t in p.terms]))

    new_conclusion = Predicate(rule.conclusion.name, [get_new_term(t) for t in rule.conclusion.terms])
    return Rule(new_premises, new_conclusion)

# Basic Unification Algorithm for FOL
def unify(x, y, theta):
    if theta is None:
        return None
    elif x == y:
        return theta
    elif isinstance(x, Term) and x.is_variable:
        return unify_var(x, y, theta)
    elif isinstance(y, Term) and y.is_variable:
        return unify_var(y, x, theta)
    elif isinstance(x, Predicate) and isinstance(y, Predicate):
        if x.name != y.name or len(x.terms) != len(y.terms):
            return None
        for t1, t2 in zip(x.terms, y.terms):
            theta = unify(t1, t2, theta)
        return theta
    else:
        return None

def unify_var(var, x, theta):
    if var.name in theta:
        return unify(theta[var.name], x, theta)
    elif isinstance(x, Term) and x.name in theta:
        return unify(var, theta[x.name], theta)
    # Occur check omitted for simple demonstration purposes
    theta = copy.deepcopy(theta)
    theta[var.name] = x
    return theta

def subst(theta, predicate):
    new_terms = []
    for t in predicate.terms:
        if t.is_variable and t.name in theta:
            resolved = theta[t.name]
            # Follow chain if needed
            while isinstance(resolved, Term) and resolved.is_variable and resolved.name in theta:
                resolved = theta[resolved.name]
            new_terms.append(resolved)
        else:
            new_terms.append(t)
    return Predicate(predicate.name, new_terms)

In [ ]:
def fol_fc_ask(KB_facts, KB_rules, query):
    """Standard First-Order Logic Forward Chaining implementation."""
    facts = set(KB_facts)
    print("--- Initial Knowledge Base --- ")
    for fact in facts:
        print(f"Fact: {fact}")
    for rule in KB_rules:
        print(f"Rule: {rule}")
    print("------------------------------\n")

    step = 1
    while True:
        new = set()
        for rule in KB_rules:
            standardized = standardize_variables(rule)

            # Find all possible substitutions that satisfy the premises
            substitutions = [{}]

            for premise in standardized.premises:
                next_substitutions = []
                for theta in substitutions:
                    # Instantiate premise with current substitution mapping
                    inst_premise = subst(theta, premise)
                    for fact in facts:
                        # Try unifying the fact with the instantiated premise
                        unif_theta = unify(inst_premise, fact, {})
                        if unif_theta is not None:
                            # Merge the new substitution mappings
                            merged_theta = {**theta, **unif_theta}
                            next_substitutions.append(merged_theta)
                substitutions = next_substitutions

            # For each valid mapping, infer the conclusion
            for theta in substitutions:
                inferred = subst(theta, standardized.conclusion)
                if inferred not in facts and inferred not in new:
                    new.add(inferred)
                    print(f"[Step {step}] Inferred new fact: {inferred}")
                    # Check if the query is reached
                    query_unif = unify(inferred, query, {})
                    if query_unif is not None:
                        print(f"\nSUCCESS: Query '{query}' proven with substitution: {query_unif}!")
                        return True

        if not new:
            print("\nFAILED: No new facts could be inferred.")
            return False

        facts.update(new)
        step += 1

In [ ]:
# @title
# Variables
p = Term('p', is_variable=True)
q = Term('q', is_variable=True)
r = Term('r', is_variable=True)
x = Term('x', is_variable=True)

# Constants
Robert = Term('Robert')
America = Term('America')
A = Term('A')
T1 = Term('T1')

# Facts (Initial KB status)
facts = [
    Predicate('Owns', [A, T1]),
    Predicate('Missile', [T1]),
    Predicate('American', [Robert]),
    Predicate('Enemy', [A, America])
]

# Rules representing implications
rules = [
    # American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p)
    Rule([Predicate('American', [p]), Predicate('Weapon', [q]), Predicate('Sells', [p, q, r]), Predicate('Hostile', [r])],
         Predicate('Criminal', [p])),

    # Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)
    Rule([Predicate('Missile', [x]), Predicate('Owns', [A, x])],
         Predicate('Sells', [Robert, x, A])),

    # Missile(x) => Weapon(x)
    Rule([Predicate('Missile', [x])],
         Predicate('Weapon', [x])),

    # Enemy(x, America) => Hostile(x)
    Rule([Predicate('Enemy', [x, America])],
         Predicate('Hostile', [x]))
]

# Target query: Criminal(Robert)
query = Predicate('Criminal', [Robert])

# Execute reasoning
fol_fc_ask(facts, rules, query)

--- Initial Knowledge Base --- 
Fact: Missile(T1)
Fact: American(Robert)
Fact: Enemy(A, America)
Fact: Owns(A, T1)
Rule: American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p)
Rule: Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)
Rule: Missile(x) => Weapon(x)
Rule: Enemy(x, America) => Hostile(x)
------------------------------

[Step 1] Inferred new fact: Sells(Robert, T1, A)
[Step 1] Inferred new fact: Weapon(T1)
[Step 1] Inferred new fact: Hostile(A)
[Step 2] Inferred new fact: Criminal(Robert)

SUCCESS: Query 'Criminal(Robert)' proven with substitution: {}!


True

### Dynamic Input Parser for FOL
Now, let's write a simple parser that reads text representations of rules and facts, converting them dynamically into our `Term`, `Predicate`, and `Rule` structures.

In [ ]:
import re

def parse_predicate(expr):
    # Matches something like PredName(arg1, arg2, ...)
    match = re.match(r"(\w+)\((.*)\)", expr.strip())
    if not match:
        raise ValueError(f"Invalid predicate format: {expr}")

    pred_name = match.group(1)
    args_str = match.group(2).strip()

    terms = []
    if args_str:
        for arg in args_str.split(','):
            arg = arg.strip()
            # Convention: if lowercase single letter/word starting with lowercase, it is a variable
            is_var = arg[0].islower() and len(arg) == 1
            terms.append(Term(arg, is_variable=is_var))

    return Predicate(pred_name, terms)

def parse_rule(expr):
    # Matches Premise1 ^ Premise2 => Conclusion
    if "=>" not in expr:
        raise ValueError(f"Rule must contain '=>': {expr}")

    premises_str, conclusion_str = expr.split("=>")

    premises = []
    for p_expr in premises_str.split("^"):
        premises.append(parse_predicate(p_expr))

    conclusion = parse_predicate(conclusion_str)
    return Rule(premises, conclusion)

def parse_input_and_run(facts_input, rules_input, query_input):
    parsed_facts = [parse_predicate(f) for f in facts_input if f.strip()]
    parsed_rules = [parse_rule(r) for r in rules_input if r.strip()]
    parsed_query = parse_predicate(query_input)

    fol_fc_ask(parsed_facts, parsed_rules, parsed_query)

In [ ]:
print("--- Enter your Problem Input (Interactive) ---")
print("Enter facts separated by semicolons (e.g., Owns(A, T1); Missile(T1)):")
facts_raw = input("Facts: ")

print("\nEnter rules separated by semicolons (e.g., Missile(x) => Weapon(x)):")
rules_raw = input("Rules: ")

query_raw = input("\nEnter query (e.g., Criminal(Robert)): ")

# Split inputs into lists
custom_facts = [f.strip() for f in facts_raw.split(";") if f.strip()]
custom_rules = [r.strip() for r in rules_raw.split(";") if r.strip()]
custom_query = query_raw.strip()

print("\n--- Running FOL Forward Chaining ---")
parse_input_and_run(custom_facts, custom_rules, custom_query)

--- Enter your Problem Input (Interactive) ---
Enter facts separated by semicolons (e.g., Owns(A, T1); Missile(T1)):
Facts: Owns(A, T1); Missile(T1); American(Robert); Enemy(A, America)

Enter rules separated by semicolons (e.g., Missile(x) => Weapon(x)):
Rules: American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p); Missile(x) ^ Owns(A, x) => Sells(Robert, x, A); Missile(x) => Weapon(x); Enemy(x, America) => Hostile(x)

Enter query (e.g., Criminal(Robert)): Criminal(Robert)

--- Running FOL Forward Chaining ---
--- Initial Knowledge Base --- 
Fact: Missile(T1)
Fact: American(Robert)
Fact: Enemy(A, America)
Fact: Owns(A, T1)
Rule: American(p) ^ Weapon(q) ^ Sells(p, q, r) ^ Hostile(r) => Criminal(p)
Rule: Missile(x) ^ Owns(A, x) => Sells(Robert, x, A)
Rule: Missile(x) => Weapon(x)
Rule: Enemy(x, America) => Hostile(x)
------------------------------

[Step 1] Inferred new fact: Sells(Robert, T1, A)
[Step 1] Inferred new fact: Weapon(T1)
[Step 1] Inferred new fact: Hostile(A